# Customer Churn Prediction and Analytics
## Phase 3: EDA, Preprocessing, and Machine Learning
This notebook fulfills the following project requirements:
- Perform Exploratory Data Analysis (EDA)
- Identify missing values and data-quality issues
- Produce at least four meaningful visualizations
- Perform suitable data preprocessing
- Develop at least two machine-learning classification models
- Evaluate and compare the models
- Identify customer characteristics associated with churn
- Store processed datasets in the cloud (S3)


In [ ]:
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Configuration: UPDATE THIS TO YOUR BUCKET NAME!
BUCKET_NAME = 'telecom-churn-project-yourname' # <--- CHANGE THIS
RAW_FILE_KEY = 'raw/lk_telecom_synthetic_100k.csv'
PROCESSED_FILE_KEY = 'processed/cleaned_telecom_data.csv'

s3 = boto3.client('s3')

## 1. Load Data from S3

In [ ]:
# Download the file from S3 to the local SageMaker instance
s3.download_file(BUCKET_NAME, RAW_FILE_KEY, 'raw_data.csv')

# Load into Pandas DataFrame
df = pd.read_csv('raw_data.csv')
print(f'Dataset shape: {df.shape}')
display(df.head())

## 2. Exploratory Data Analysis & Data Quality Checks

In [ ]:
# Check for missing values
missing_data = df.isnull().sum()
print("Missing Values:\n", missing_data[missing_data > 0])

# Note: Missing 'device_type' and 'device_brand' is intentional per data rules.
# Check data types
print("\nData Types:\n", df.dtypes)

## 3. Data Visualizations (4 Required)

In [ ]:
# Visualization 1: Churn Rate
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='churned', palette='Set2')
plt.title('Distribution of Churned vs Active Customers')
plt.show()

# Visualization 2: Churn by Operator
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='operator', hue='churned', palette='Set1')
plt.title('Customer Churn by Telecom Operator')
plt.show()

# Visualization 3: Contract Type vs Churn
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='contract_type', hue='churned', palette='viridis')
plt.title('Customer Churn by Contract Type')
plt.show()

# Visualization 4: Days Since Last Recharge distribution for Churned vs Non-Churned (Prepaid only)
prepaid_df = df[df['contract_type'] == 'prepaid'].dropna(subset=['days_since_last_recharge'])
plt.figure(figsize=(10, 6))
sns.kdeplot(data=prepaid_df, x='days_since_last_recharge', hue='churned', fill=True, common_norm=False, palette='crest')
plt.title('Density of Days Since Last Recharge (Prepaid)')
plt.show()

## 4. Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df_clean = df.copy()

# 4.1 Handle Missing Values
df_clean['device_type'] = df_clean['device_type'].fillna('Unknown')
df_clean['device_brand'] = df_clean['device_brand'].fillna('Unknown')
df_clean['days_since_last_recharge'] = df_clean['days_since_last_recharge'].fillna(-1)
df_clean['late_fee_lkr'] = df_clean['late_fee_lkr'].fillna(0)
df_clean['overdue_days'] = df_clean['overdue_days'].fillna(0)

# Drop date and text columns
cols_to_drop = ['customer_id', 'signup_date', 'subscription_activated_at', 'last_recharge_date', 'last_bill_date', 'complaint_text', 'bundled_apps']
df_clean = df_clean.drop(columns=cols_to_drop)

# 4.2 Encode Categorical Variables
label_encoders = {}
categorical_cols = df_clean.select_dtypes(include=['object', 'bool']).columns
if 'churned' in categorical_cols:
    categorical_cols = categorical_cols.drop('churned')

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le

df_clean['churned'] = df_clean['churned'].astype(int)

# 4.3 Save processed data to local, then upload to S3
df_clean.to_csv('processed_data.csv', index=False)
s3.upload_file('processed_data.csv', BUCKET_NAME, PROCESSED_FILE_KEY)
print(f'Processed data uploaded to s3://{BUCKET_NAME}/{PROCESSED_FILE_KEY}')

# 4.4 Split Data
X = df_clean.drop('churned', axis=1)
y = df_clean['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4.5 Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Machine Learning Modeling (2 Models)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
lr_model = LogisticRegression(max_iter=1000, random_state=42)

print("Training Random Forest...")
rf_model.fit(X_train_scaled, y_train)

print("Training Logistic Regression...")
lr_model.fit(X_train_scaled, y_train)

## 6. Model Evaluation and Comparison

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    print(f"\n--- {model_name} Evaluation ---")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {model_name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

evaluate_model(rf_model, X_test_scaled, y_test, "Random Forest")
evaluate_model(lr_model, X_test_scaled, y_test, "Logistic Regression")

## 7. Identify Characteristics Associated with Churn

In [ ]:
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importances.head(15), x='Importance', y='Feature', palette='magma')
plt.title('Top 15 Characteristics Associated with Churn')
plt.show()